# Early stopping
A small network on 100 points of `make_circles`: trained for 3,500 epochs it overfits; with the `EarlyStopping`
callback it stops by itself near the best epoch. Results go to `data/` for the figures in `images/`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

tf.config.experimental.enable_op_determinism()   # same numbers on every run

## The data: two noisy circles

In [ ]:
# 100 points: an inner circle (class 1) inside an outer one (class 0), with noise
X, y = make_circles(n_samples=100, noise=0.1, random_state=1)
# half for training, half for validation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=2)
print(X_train.shape, X_test.shape, np.bincount(y_train))

## The network
One hidden layer of 256 ReLU nodes, one sigmoid output node.

In [ ]:
def build(seed=0):
    keras.utils.set_random_seed(seed)          # same starting weights every time
    model = keras.Sequential([
        keras.Input(shape=(2,)),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

build().summary()

## 1. Training for 3,500 epochs
This takes a few minutes: every epoch also scores the validation set.

In [ ]:
model = build()
history = model.fit(X_train, y_train, epochs=3500, validation_data=(X_test, y_test), verbose=0)
h = pd.DataFrame(history.history)
h.index += 1                                   # epochs counted from 1
best = h["val_loss"].idxmin()
print(f"lowest validation loss {h.val_loss.min():.3f} at epoch {best}")
print(h.loc[[1, best, 1000, 2000, 3500]].round(3))

## 2. The same network with early stopping

In [ ]:
from keras.callbacks import EarlyStopping

callback = EarlyStopping(
    monitor="val_loss",       # watch the validation loss
    min_delta=0.00001,        # smaller changes do not count as improvement
    patience=50,              # wait 50 epochs without improvement
    verbose=1,                # print the epoch at which it stops
    mode="auto",              # loss: lower is better (worked out from the name)
    baseline=None,
    restore_best_weights=False,
)
model_es = build()
history_es = model_es.fit(X_train, y_train, epochs=3500, validation_data=(X_test, y_test),
                          verbose=0, callbacks=[callback])
h_es = pd.DataFrame(history_es.history)
h_es.index += 1
best_es = h_es["val_loss"].idxmin()
print("epochs run:", len(h_es), "| best epoch:", best_es, "| best val_loss:", round(h_es.val_loss.min(), 3))
print(h_es.tail(1).round(3))

## Comparing the two models on the validation data

In [ ]:
for name, m in [("3,500 epochs", model), ("early stopping", model_es)]:
    loss, acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"{name:15s} val_loss {loss:.3f}  val_accuracy {acc:.2f}")

## 3. restore_best_weights=True
Same run, but at the end Keras puts back the weights of the best epoch.

In [ ]:
model_rb = build()
model_rb.fit(X_train, y_train, epochs=3500, validation_data=(X_test, y_test), verbose=0,
             callbacks=[EarlyStopping(monitor="val_loss", min_delta=0.00001, patience=50,
                                      restore_best_weights=True)])
loss, acc = model_rb.evaluate(X_test, y_test, verbose=0)
print(f"restored weights: val_loss {loss:.3f}  val_accuracy {acc:.2f}")

## 4. Too little patience
With patience 20, a small rise of the validation loss in the first epochs is enough to stop training.

In [ ]:
model_p20 = build()
h_p20 = model_p20.fit(X_train, y_train, epochs=3500, validation_data=(X_test, y_test), verbose=0,
                      callbacks=[EarlyStopping(monitor="val_loss", min_delta=0.00001, patience=20)]).history
loss, acc = model_p20.evaluate(X_test, y_test, verbose=0)
print("epochs run:", len(h_p20["loss"]), f"| val_loss {loss:.3f}  val_accuracy {acc:.2f}")
print("validation loss, epochs 1 to 21:", np.round(h_p20["val_loss"], 4))

## Saving results for the figures

In [ ]:
os.makedirs("data", exist_ok=True)
h.to_csv("data/history_3500.csv", index_label="epoch")
h_es.to_csv("data/history_early_stopping.csv", index_label="epoch")

# decision boundaries: predicted probability on a grid
xx, yy = np.meshgrid(np.linspace(-1.6, 1.6, 161), np.linspace(-1.6, 1.6, 161))
grid = np.c_[xx.ravel(), yy.ravel()]
pd.DataFrame({"x1": grid[:, 0], "x2": grid[:, 1],
              "p_3500": model.predict(grid, verbose=0).ravel(),
              "p_es": model_es.predict(grid, verbose=0).ravel()}).round(4).to_csv("data/boundary_grid.csv", index=False)
pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y,
              "set": ["train" if any((X_train == r).all(1)) else "validation" for r in X]}).to_csv("data/points.csv", index=False)
print("saved")